# TP4 — Clustering non supervisé avec K-Means
**Module :** Intelligence Artificielle  
**Filière :** MSEI — ENSAM Casablanca  
**Année universitaire :** 2026–2027

Objectif : appliquer l'algorithme **K-Means** sur un jeu de données simulées représentant le fonctionnement quotidien d'un atelier industriel, afin d'identifier différents profils de fonctionnement.

## Étape 1 — Charger et explorer le jeu de données
**Objectif :** importer le jeu de données, examiner sa structure et identifier les caractéristiques nécessaires au clustering.

In [ ]:
# Importer les bibliothèques nécessaires
import pandas as pd
from google.colab import files

# Importer le fichier CSV
uploaded = files.upload()

In [ ]:
# Charger le jeu de données
df = pd.read_csv("TP4_Clustering_MSEI.csv")

# Afficher les cinq premières observations
df.head()

In [ ]:
# Afficher les dimensions du jeu de données
print("Dimensions :", df.shape)

# Afficher les noms des colonnes
print("\nColonnes :")
print(df.columns)

# Afficher les informations générales
print("\nInformations sur les données :")
df.info()

# Vérifier les valeurs manquantes
print("\nTotal des valeurs manquantes :")
print(df.isnull().sum().sum())

### Réponses aux questions — Étape 1
1. **Que représente une ligne et combien d'observations ?** Chaque ligne représente une journée de fonctionnement de l'atelier industriel. Le jeu de données contient **500 observations**.
2. **Quelles sont les quatre caractéristiques ?** La température ambiante (°C), le nombre de machines actives, la durée de fonctionnement (heures) et le taux de charge (%).
3. **Pourquoi pas de variable cible (y) ?** Parce que le clustering est une méthode d'apprentissage **non supervisé** : on ne connaît pas les catégories à l'avance, le modèle doit les découvrir automatiquement.

## Étape 2 — Identifier et standardiser les variables (X)
**Objectif :** identifier les quatre caractéristiques et standardiser leurs valeurs (moyenne ≈ 0, écart-type ≈ 1) avec `StandardScaler`, car K-Means utilise des distances.

In [ ]:
# Importer la méthode de standardisation
from sklearn.preprocessing import StandardScaler

# Identifier les variables utilisées pour le clustering
X = df[[
    "Temperature_C",
    "Machines_actives",
    "Duree_fonctionnement_h",
    "Taux_charge_pct"
]]

# Afficher les dimensions
print("Dimensions de X :", X.shape)

# Créer l'outil de standardisation
scaler = StandardScaler()

# Standardiser les données
X_scaled = scaler.fit_transform(X)

# Afficher les dimensions des données standardisées
print("Dimensions de X_scaled :", X_scaled.shape)

# Afficher les cinq premières observations standardisées
print("\nPremières observations standardisées :")
print(X_scaled[:5])

### Réponses aux questions — Étape 2
1. **Pourquoi standardiser avant K-Means ?** K-Means repose sur les distances entre observations. Les variables ont des unités et échelles différentes (ex. taux de charge ~ 0-100 vs nombre de machines ~ 1-12) : sans standardisation, les variables aux grandes valeurs domineraient le calcul des distances.
2. **Rôle de StandardScaler() ?** Il transforme chaque variable pour obtenir une moyenne proche de 0 et un écart-type proche de 1.
3. **Différence entre X et X_scaled ?** X contient les valeurs originales dans leurs unités ; X_scaled contient les mêmes observations transformées (valeurs positives = au-dessus de la moyenne, négatives = en dessous), avec les mêmes dimensions (500, 4).

## Étape 3 — Déterminer le nombre de clusters (K) avec la méthode du coude
**Objectif :** entraîner K-Means pour K de 1 à 10, mesurer l'inertie et repérer le « coude » sur la courbe.

In [ ]:
# Importer les bibliothèques nécessaires
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Créer une liste pour enregistrer les inerties
inerties = []

# Tester différentes valeurs de K
for k in range(1, 11):
    # Créer le modèle K-Means
    modele = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    # Entraîner le modèle
    modele.fit(X_scaled)
    # Enregistrer l'inertie
    inerties.append(modele.inertia_)

# Tracer la courbe de la méthode du coude
plt.figure(figsize=(8, 6))
plt.plot(range(1, 11), inerties, marker='o')
plt.xlabel("Nombre de clusters (K)")
plt.ylabel("Inertie")
plt.title("Méthode du coude (Elbow Method)")
plt.grid(True)
plt.show()

### Réponses aux questions — Étape 3
1. **Que représente K ?** Le nombre de groupes (clusters) que l'algorithme doit former.
2. **Objectif de la méthode du coude ?** Aider à choisir un nombre de clusters adapté en repérant le point où l'ajout de clusters supplémentaires n'améliore plus significativement l'inertie.
3. **Nombre de clusters approprié ?** **K = 3** : l'inertie diminue fortement de K=1 à K=3 (2000 → 676 → 335), puis la diminution devient faible. La courbe présente un coude autour de K = 3.

## Étape 4 — Créer et entraîner un modèle K-Means
**Objectif :** entraîner un modèle K-Means avec K = 3 et afficher les centroïdes.

In [ ]:
from sklearn.cluster import KMeans

# Créer le modèle
modele = KMeans(n_clusters=3, random_state=42, n_init=10)

# Entraîner le modèle
modele.fit(X_scaled)

# Afficher les centroïdes
print("Centroïdes des clusters :")
print(modele.cluster_centers_)

### Réponses aux questions — Étape 4
1. **Rôle de K-Means ?** Regrouper automatiquement les observations similaires en K clusters, en associant chaque observation au centroïde le plus proche.
2. **Que signifie n_clusters=3 ?** Le modèle doit répartir les 500 observations en 3 groupes.
3. **Qu'est-ce qu'un centroïde ?** Le centre d'un cluster : la moyenne des caractéristiques des observations qui lui appartiennent.

## Étape 5 — Attribuer un cluster à chaque observation
**Objectif :** ajouter le numéro de cluster à chaque observation et compter les effectifs de chaque groupe.

In [ ]:
# Ajouter les clusters au jeu de données
df["Cluster"] = modele.labels_

# Afficher les 10 premières observations
display(df.head(10))

# Compter les observations par cluster
print("Nombre d'observations par cluster :")
print(df["Cluster"].value_counts().sort_index())

### Réponses aux questions — Étape 5
1. **Rôle de modele.labels_ ?** Il contient le numéro de cluster (0, 1 ou 2) attribué à chaque observation après l'entraînement.
2. **Que représente la colonne Cluster ?** Le groupe auquel appartient chaque journée de fonctionnement selon le modèle.
3. **Les clusters ont-ils le même nombre d'observations ?** Non, pas nécessairement : K-Means forme des groupes selon les distances, pas selon des effectifs égaux (ici : 171, 165 et 164 observations).

## Étape 6 — Visualiser les clusters
**Objectif :** représenter les observations (machines actives × taux de charge), colorées par cluster.

In [ ]:
import matplotlib.pyplot as plt

# Créer le graphique
plt.figure(figsize=(8, 6))

# Afficher les observations selon leur cluster
plt.scatter(
    df["Machines_actives"],
    df["Taux_charge_pct"],
    c=df["Cluster"],
    cmap="viridis",
    alpha=0.7
)

# Ajouter les titres
plt.xlabel("Nombre de machines actives")
plt.ylabel("Taux de charge (%)")
plt.title("Visualisation des clusters K-Means")

# Afficher la légende des couleurs
plt.colorbar(label="Cluster")
plt.grid(True)
plt.show()

### Réponses aux questions — Étape 6
1. **Que représente chaque point ?** Une journée de fonctionnement de l'atelier industriel.
2. **Pourquoi différentes couleurs ?** Chaque couleur correspond à un cluster (0, 1 ou 2) attribué par K-Means.
3. **Pourquoi des chevauchements possibles ?** K-Means utilise les **quatre** caractéristiques pour former les clusters, alors que le graphique n'en représente que **deux** : des points de clusters différents peuvent donc apparaître proches.

## Étape 7 — Interpréter les caractéristiques des clusters
**Objectif :** calculer les moyennes des quatre variables par cluster pour identifier les profils de fonctionnement.

In [ ]:
# Calculer les moyennes de chaque cluster
profils = df.groupby("Cluster").mean(numeric_only=True)

# Afficher les résultats
print("Caractéristiques moyennes des clusters :")
display(profils.round(2))

### Réponses aux questions — Étape 7
1. **Rôle de groupby("Cluster") ?** Regrouper les observations selon leur numéro de cluster afin de calculer des statistiques (ici la moyenne) par groupe.
2. **Quel cluster a le taux de charge moyen le plus élevé ?** Le **cluster 1** (≈ 84,33 %) — profil de **forte activité**.
3. **Comment ces résultats aident-ils l'analyse d'un atelier ?** Ils identifient trois profils de fonctionnement : **faible activité** (cluster 0 : ~2,9 machines, 5,1 h, 30,1 %), **forte activité** (cluster 1 : ~11,9 machines, 14,6 h, 84,3 %) et **activité moyenne** (cluster 2 : ~7 machines, 8,9 h, 60 %). Les responsables peuvent ainsi comprendre les modes d'utilisation des équipements et planifier la production ou la maintenance.

## Étape 8 — Évaluer la qualité du clustering avec le score de silhouette
**Objectif :** mesurer la qualité du regroupement (score entre -1 et 1).

In [ ]:
# Importer la métrique d'évaluation
from sklearn.metrics import silhouette_score

# Calculer le score de silhouette
score = silhouette_score(X_scaled, modele.labels_)

# Afficher le résultat
print("Score de silhouette :", round(score, 3))

### Réponses aux questions — Étape 8
1. **Rôle du score de silhouette ?** Mesurer si les observations sont proches de celles de leur propre cluster et bien éloignées des autres clusters.
2. **Score proche de 1 ?** Les clusters sont bien séparés et cohérents.
3. **Pourquoi évaluer la qualité des clusters ?** Pour vérifier que le regroupement obtenu est pertinent et compléter l'analyse visuelle — un mauvais score indiquerait des groupes peu distincts.

## Conclusion
Ce TP a permis d'appliquer les étapes du clustering non supervisé : exploration des données, standardisation, choix de K par la méthode du coude (K = 3), entraînement de K-Means, visualisation et interprétation des clusters, puis évaluation par le score de silhouette (**0,526**).

Trois profils de fonctionnement industriel ont été identifiés : **faible activité**, **activité moyenne** et **forte activité**.